In [ ]:
!pip install git+https://github.com/sberbank-ai/Real-ESRGAN.git

In [ ]:
import os
import shutil
from PIL import Image, UnidentifiedImageError
import numpy as np
from RealESRGAN import RealESRGAN
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

model_scale = "4"  # Choose the model scale (2x, 4x, 8x)
model = RealESRGAN(device, scale=int(model_scale))
model.load_weights(f'weights/RealESRGAN_x{model_scale}.pth')

# Bitmap formats supported by PIL and RealESRGAN
BITMAP_FORMATS = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff", ".tif", ".webp", ".heic", ".heif", ".ico", ".avif")

# All other formats to be recognized but skipped for processing
ALL_FORMATS = BITMAP_FORMATS + (
    # Vector Image Formats
    ".svg", ".eps", ".ai", ".pdf", ".cdr",

    # 3D Image Formats
    ".obj", ".stl", ".fbx", ".dae", ".3ds",

    # Specialized and Less Common Formats
    ".xpm", ".pict", ".pct", ".dds", ".hdr", ".exr", ".tga", ".sgi",
    ".wbmp", ".pcx", ".ppm", ".pgm", ".pbm", ".xbm",

    # Legacy and Proprietary Formats
    ".img", ".vda", ".icns", ".jbig", ".jb2", ".afphoto", ".psd", ".raw", ".cr2", ".nef", ".arw", ".orf",
    ".rw2", ".dng"
)

MAX_SIZE = (500, 500)  # Maximum image dimensions before upscaling is skipped

def upscale_image(image_path, output_path):
    try:
        image = Image.open(image_path).convert('RGB')
        if image.size[0] <= MAX_SIZE[0] and image.size[1] <= MAX_SIZE[1]:
            image_np = np.array(image)  # Convert the image to a numpy array
            sr_image_np = model.predict(image_np)  # Apply the model to the numpy array
            sr_image = Image.fromarray(np.uint8(sr_image_np))  # Convert the numpy array back to a PIL image
            sr_image.save(output_path)
            print(f'Upscaled and saved: {output_path}')
        else:
            shutil.copy2(image_path, output_path)
            print(f'Copied without upscaling: {output_path}')
    except (UnidentifiedImageError, Exception) as e:
        # If any error occurs, copy the file without processing
        shutil.copy2(image_path, output_path)
        print(f"Copied unsupported or unprocessable file without processing: {image_path}")

def process_directory(input_dir, output_dir):
    for root, dirs, files in os.walk(input_dir):
        # Create corresponding directory structure in the output directory
        relative_path = os.path.relpath(root, input_dir)
        output_root = os.path.join(output_dir, relative_path)
        os.makedirs(output_root, exist_ok=True)

        for file in files:
            input_path = os.path.join(root, file)
            output_path = os.path.join(output_root, file)

            if file.lower().endswith(BITMAP_FORMATS):
                upscale_image(input_path, output_path)
            elif file.lower().endswith(ALL_FORMATS):
                shutil.copy2(input_path, output_path)
                print(f"Copied unsupported format without processing: {file}")
            elif file.lower() == "images111.csv":
                shutil.copy2(input_path, output_path)
                print(f"Copied CSV file: {file}")
            else:
                print(f"Skipping non-image file: {file}")

# Input and output directories
input_root_dir = '/content/Your Input Folder'
output_root_dir = '/content/drive/MyDrive/Your Output Folder'

process_directory(input_root_dir, output_root_dir)
print(f'Finished processing all images from {input_root_dir} to {output_root_dir}')